# Validación de QuEtool — Vina nativo como contraste

Corre **AutoDock Vina nativo 1.2.3** (la misma versión que lleva QuEtool compilada a
WebAssembly, *Webina*) sobre los mismos once ligandos, el mismo receptor y la misma caja.

Dos funciones dentro de la validación:

1. **Tapa el hueco del protocolo**: una corrida nativa e independiente para contrastar los ΔG.
2. **Prueba que el port a WebAssembly no se desvía del binario oficial.**

Corre en Colab gratis, en **CPU** (no hace falta GPU). La salida queda en
`QuEtool - materiales TP reconstruidos/Validacion-Vina-nativo/`.


## 1. Montar el Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Encontrar la carpeta de materiales

Mira **sólo la raíz** de Mi unidad: es instantáneo. (No hace una búsqueda recursiva por
todo el Drive, que es lo que colgaba la versión anterior.)


In [ ]:
import os, glob
NOMBRE = 'QuEtool - materiales TP reconstruidos'

directo = os.path.join('/content/drive/MyDrive', NOMBRE)
BASE = directo if os.path.isdir(directo) else None
if BASE is None:                      # por si estuviera una carpeta mas adentro
    for d in glob.glob('/content/drive/MyDrive/*'):
        if os.path.basename(d) == NOMBRE and os.path.isdir(d):
            BASE = d; break
if BASE is None:
    print('No la encontre sola. Carpetas en la raiz de tu Drive:')
    for d in sorted(glob.glob('/content/drive/MyDrive/*')):
        if os.path.isdir(d): print('   -', os.path.basename(d))
    raise SystemExit('Edita BASE a mano (una linea mas arriba) con la ruta correcta.')

OUT = os.path.join(BASE, 'Validacion-Vina-nativo')
os.makedirs(os.path.join(OUT, 'poses'), exist_ok=True)
print('Materiales:', BASE)
print('Receptor  :', os.path.exists(os.path.join(BASE, 'Receptor', '2B6A_receptor.pdbqt')))
print('Ligandos  :', len(glob.glob(os.path.join(BASE, 'Ligandos', '*.pdbqt'))), 'PDBQT')
print('Salida    :', OUT)


## 3. Bajar Vina nativo 1.2.3

La misma versión que Webina, para que la comparación sea exacta. Binario estático oficial.


In [ ]:
import subprocess, urllib.request, os
URL = 'https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.3/vina_1.2.3_linux_x86_64'
VINA = '/content/vina'
if not os.path.exists(VINA):
    urllib.request.urlretrieve(URL, VINA); os.chmod(VINA, 0o755)
print(subprocess.run([VINA, '--version'], capture_output=True, text=True).stdout.strip())


## 4. Leer la caja del `conf.txt`

Se lee del mismo `conf.txt` que usa QuEtool: la caja queda idéntica por construcción.


In [ ]:
conf = {}
with open(os.path.join(BASE, 'conf.txt')) as f:
    for ln in f:
        ln = ln.split('#')[0].strip()
        if '=' in ln:
            k, v = ln.split('='); conf[k.strip()] = float(v.strip())
CAJA = {k: conf[k] for k in ('center_x','center_y','center_z','size_x','size_y','size_z')}
EXH, SEED = int(conf.get('exhaustiveness', 10)), 42
print('Caja:', CAJA); print('Exhaustiveness:', EXH, '| seed:', SEED)


## 5. Dockear los once ligandos

Receptor rígido, misma caja, misma exhaustiveness, semilla fija. Guarda cada pose y se
queda con el ΔG de la mejor.


In [ ]:
import glob, os, time
REC = os.path.join(BASE, 'Receptor', '2B6A_receptor.pdbqt')
dG_nativo = {}
for lp in sorted(glob.glob(os.path.join(BASE, 'Ligandos', '*.pdbqt'))):
    nom = os.path.splitext(os.path.basename(lp))[0]
    out = os.path.join(OUT, 'poses', f'{nom}_vina.pdbqt')
    cmd = [VINA,'--receptor',REC,'--ligand',lp,'--out',out,
           '--exhaustiveness',str(EXH),'--seed',str(SEED),'--cpu','2']
    for k, v in CAJA.items(): cmd += ['--'+k, str(v)]
    t0 = time.time(); subprocess.run(cmd, capture_output=True, text=True)
    dg = None
    with open(out) as f:
        for ln in f:
            if 'VINA RESULT' in ln: dg = float(ln.split(':')[1].split()[0]); break
    dG_nativo[nom] = dg
    print(f'{nom:9s}  dG = {dg:7.2f} kcal/mol   ({time.time()-t0:4.0f} s)')


## 6. Tabla final: nativo vs Webina, con pIC50

Los ΔG de Webina se completan en `WEBINA` con la corrida de QuEtool (hoy sólo el control T50).


In [ ]:
import csv, math
WEBINA = {'T50': -12.501}   # se completa con la corrida de QuEtool
IC50 = {'GFA':2.00,'EFZ':6.03,'TBO':14.13,'TNK':40.74,'NVP':89.13,'HBQ':177.83,
        'QNO47b':213.80,'U05':398.11,'ZZE':630.96,'QNO47c':912.01}
pIC50 = lambda nM: 9 - math.log10(nM)
filas = []
for nom in sorted(dG_nativo):
    dn, dw = dG_nativo[nom], WEBINA.get(nom)
    filas.append(dict(ligando=nom, dG_vina_nativo=dn, dG_webina=dw,
                      delta=(None if dw is None else round(dn-dw,3)),
                      IC50_nM=IC50.get(nom), pIC50=(round(pIC50(IC50[nom]),3) if nom in IC50 else None)))
csv_path = os.path.join(OUT, 'dG_vina_nativo_vs_webina.csv')
with open(csv_path,'w',newline='') as f:
    w = csv.DictWriter(f, fieldnames=['ligando','dG_vina_nativo','dG_webina','delta','IC50_nM','pIC50'])
    w.writeheader(); w.writerows(filas)
print('CSV:', csv_path, '\n')
fmt = lambda x,p='.2f': ('—' if x is None else format(x,p))
print(f"{'ligando':9s} {'nativo':>8s} {'webina':>8s} {'delta':>7s} {'IC50':>9s} {'pIC50':>6s}")
for r in filas:
    print(f"{r['ligando']:9s} {fmt(r['dG_vina_nativo']):>8s} {fmt(r['dG_webina']):>8s} "
          f"{fmt(r['delta'],'.3f'):>7s} {fmt(r['IC50_nM'],'.2f'):>9s} {fmt(r['pIC50'],'.3f'):>6s}")


## 7. Qué mirar

- **`delta`** (nativo − Webina) chica en los once = el Vina de WebAssembly no se desvía
  del oficial: ésa es la prueba.
- **T50** es el control; su ΔG nativo debería caer cerca del −12,501 de Webina.
- Con `pIC50` y los ΔG ya se ajusta `pIC50 = f(ΔG)` y se contesta la pregunta del TP.

El CSV y las poses quedan en `Validacion-Vina-nativo/` para que el análisis los levante.
